# Deteksi pohon sawit dengan YOLO di Colab, 3 ortofoto

Notebook ini menjalankan alur dari `mwahyur46/oilpalm-tree-detection-yolov26` (data prep → train/eval → inference) memakai skrip ringan di folder `yolo_oilpalm/`.

**Yang perlu ada di Google Drive** (folder `MyDrive/oilpalm/`):

```
oilpalm/
├── ORTO271026.tif                  0.155 m/px, EPSG:32648   (sumber label)
├── PMBN_ORTO1.tif                  0.099 m/px
├── PMBN_ORTO2.tif                  0.071 m/px, tertulis 48S tapi aslinya 48N
├── output/palm_crowns.gpkg         13.983 poligon tajuk -> label bounding box
└── yolo_oilpalm/                   make_tiles.py, train.py, predict.py, render_points.py
```

Label hanya tersedia untuk **ORTO271026** (hasil digitasi). ORTO1/ORTO2 diprediksi dengan model yang dilatih dari ORTO26, itu intinya: satu model, tiga ortofoto.

Runtime GPU T4: dataset ±25 detik, training 60 epoch ±10–15 menit, prediksi tiga ortofoto ±5–10 menit.

In [ ]:
# 1) Dependensi
!pip install -q "ultralytics>=8.3.206"
try:
    from osgeo import gdal
except ImportError:
    !apt-get -qq update > /dev/null && apt-get -qq install -y gdal-bin python3-gdal
    from osgeo import gdal

import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| CUDA", torch.cuda.is_available(), "| GDAL", gdal.VersionInfo())

In [ ]:
# 2) Hubungkan Drive dan cek berkas
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
BASE = "/content/drive/MyDrive/oilpalm"          # ubah kalau foldernya lain
CODE = f"{BASE}/yolo_oilpalm"
ORTHOS = {
    "ORTO271026": (f"{BASE}/ORTO271026.tif", None),           # 0.155 m, EPSG:32648
    "PMBN_ORTO1": (f"{BASE}/PMBN_ORTO1.tif", None),           # 0.099 m
    "PMBN_ORTO2": (f"{BASE}/PMBN_ORTO2.tif", "EPSG:32648"),   # salah tag 48S
}
CROWNS = f"{BASE}/output/palm_crowns.gpkg"

for k, (p, srs) in ORTHOS.items():
    print(f"{k:12s} ada={os.path.exists(p)}  crs_override={srs}")
print("crowns  ada=", os.path.exists(CROWNS))
print("scripts ada=", all(os.path.exists(f"{CODE}/{f}") for f in
                          ("make_tiles.py", "train.py", "predict.py", "render_points.py")))

In [ ]:
# 3) Cek GSD / CRS / perkiraan jumlah tile tiap ortofoto
import math
for k, (p, srs) in ORTHOS.items():
    if not os.path.exists(p):
        continue
    ds = gdal.Open(p)
    gt = ds.GetGeoTransform(); gsd = abs(gt[1])
    W, H = ds.RasterXSize, ds.RasterYSize
    win = max(64, int(round(640 * 0.155138 / gsd)))
    n = math.ceil(W / win) * math.ceil(H / win)
    print(f"{k}: {W}x{H} px @ {gsd:.4f} m/px | window {win} px | tile maks ~{n}")
    ds = None

In [ ]:
# 4) Dataset YOLO dari ORTO271026 + label digitasi
subprocess.run(["python3", f"{CODE}/make_tiles.py",
                "--ortho", ORTHOS["ORTO271026"][0],
                "--crowns", CROWNS,
                "--out", f"{BASE}/dataset", "--tile", "640"], check=True)

In [ ]:
# 5) Training. Model COCO sebagai awal; ganti dengan bobot sawit kalau sudah ada.
subprocess.run(["python3", f"{CODE}/train.py",
                "--model", "yolo11n.pt",
                "--data", f"{BASE}/dataset/data.yaml",
                "--epochs", "60", "--imgsz", "640", "--batch", "16",
                "--device", "0", "--project", f"{BASE}/runs", "--name", "oilpalm"],
               check=True)
WEIGHTS = f"{BASE}/runs/oilpalm/weights/best.pt"
print("bobot:", WEIGHTS, os.path.exists(WEIGHTS))

In [ ]:
# 6) Tes cepat dulu (8 tile) supaya tidak menunggu lama kalau ada masalah
subprocess.run(["python3", f"{CODE}/predict.py",
                "--ortho", ORTHOS["PMBN_ORTO2"][0], "--weights", WEIGHTS,
                "--out", f"{BASE}/hasil/tes_ORTO2", "--device", "0",
                "--assume-srs", "EPSG:32648", "--max-tiles", "8"], check=False)

In [ ]:
# 7) Prediksi ketiga ortofoto (GSD diseragamkan ke 0.155 m)
OUT = f"{BASE}/hasil"
os.makedirs(OUT, exist_ok=True)
for name, (path, srs) in ORTHOS.items():
    if not os.path.exists(path):
        print("lewati (tidak ada):", name); continue
    cmd = ["python3", f"{CODE}/predict.py", "--ortho", path, "--weights", WEIGHTS,
           "--out", f"{OUT}/{name}_pohon", "--device", "0",
           "--target-gsd", "0.155138", "--conf", "0.25", "--overlap", "96"]
    if srs:
        cmd += ["--assume-srs", srs]
    print(">>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=False)

In [ ]:
# 8) Ringkasan + zip hasil
import pandas as pd
rows = []
for name in ORTHOS:
    p = f"{OUT}/{name}_pohon.csv"
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p)
    rows.append({"orto": name, "pohon": len(df),
                 "conf_min": round(df.conf.min(), 3),
                 "conf_median": round(df.conf.median(), 3),
                 "box_m2_median": round((df.box_w_m * df.box_h_m).median(), 1)})
print(pd.DataFrame(rows).to_string(index=False))

import glob, zipfile
zpath = f"{OUT}/pohon_yolo.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*_pohon.csv") + glob.glob(f"{OUT}/*_pohon.geojson") + glob.glob(f"{OUT}/*_pohon.gpkg"):
        z.write(f, os.path.basename(f))
print("zip:", zpath)

In [ ]:
# 9) QA visual: titik deteksi di atas citra resolusi penuh (blok padat ORTO26)
subprocess.run(["python3", f"{CODE}/render_points.py",
                "--ortho", ORTHOS["ORTO271026"][0],
                "--points", f"{OUT}/ORTO271026_pohon.csv",
                "--crop", "17447", "10806", "300", "300",
                "--out", f"{OUT}/qa_ORTO271026.jpg"], check=False)
from IPython.display import Image, display
display(Image(filename=f"{OUT}/qa_ORTO271026.jpg"))

## Catatan penting

* **Bobot tidak ada di repo GitHub** (`model/` 404). Kalau nanti bobot sawit dari penulis/Zenodo sudah tersedia, lewati sel training: set `WEIGHTS` ke file itu dan langsung jalankan sel prediksi.
* **Label = digitasi kita sendiri.** Model akan meniru hasil digitasi, termasuk kesalahannya; ia bukan pengukur akurasi independen. Untuk evaluasi jujur, sisihkan satu blok geografis dari training.
* **GSD diseragamkan.** PMBN_ORTO2 0.071 m/px: tanpa `--target-gsd`, pohon tampak 2x lebih besar dari skala training dan deteksi bisa kacau.
* **PMBN_ORTO2** bertanda UTM 48S; `--assume-srs EPSG:32648` membuat titiknya keluar di 0°21′ N (sesuai posisi sebenarnya).
* Hasil prediksi tetap perlu dicek manual di QGIS; filter 5 m hanya membuang duplikat di batas tile.